In [ ]:
# Import Libraries
import numpy as np
import pandas as pd
import cv2
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from tqdm.auto import tqdm
import timm
import matplotlib.pyplot as plt
# Print PyTorch version and set device
print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Path to train metadata
train_csv_path = '/kaggle/input/grand-xray-slam-division-a/train1.csv'  # updated if different

train_df = pd.read_csv(train_csv_path)
print(f"Loaded {train_csv_path} with {len(train_df)} rows")

# Define 14 labels
label_columns = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Enlarged Cardiomediastinum',
    'Fracture', 'Lung Lesion', 'Lung Opacity', 'No Finding', 'Pleural Effusion',
    'Pleural Other', 'Pneumonia', 'Pneumothorax', 'Support Devices'
]

# Train/validation split
train_data, val_data = train_test_split(
    train_df, test_size=0.2, random_state=42, stratify=train_df['No Finding']
)
print(f"Train samples: {len(train_data)}, Validation samples: {len(val_data)}")


In [ ]:
# Jupyter cell: plot Pearson correlation matrix for the 14 diseases
# Assumes `train_df` is already loaded (as in your snippet) and `label_columns` defined.
# If not, uncomment the first two lines to load here.

# from sklearn.model_selection import train_test_split
# train_df = pd.read_csv('/kaggle/input/grand-xray-slam-division-a/train1.csv')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- configuration ---
OUT_FIG = "/kaggle/working/disease_corr_matrix.png"   # path to save figure
figsize = (12, 10)
annot_fmt = ".2f"   # annotation format for correlation values

# --- ensure label columns exist in the DataFrame ---
missing = [c for c in label_columns if c not in train_df.columns]
if len(missing) > 0:
    raise KeyError(f"The following label columns are missing from train_df: {missing}")

# --- build label dataframe (ensure binary numeric) ---
labels_df = train_df[label_columns].copy()

# If labels are not numeric (e.g. strings '0'/'1'), try to convert
for c in labels_df.columns:
    if labels_df[c].dtype not in [np.int64, np.int32, np.float32, np.float64, bool]:
        # try convert
        labels_df[c] = pd.to_numeric(labels_df[c], errors="coerce")

# If any NaNs appeared after conversion, fill with 0 (or raise depending on preference)
if labels_df.isna().any().any():
    n_na = labels_df.isna().sum().sum()
    print(f"[warning] {n_na} missing values found in label columns after conversion -> filling with 0")
    labels_df = labels_df.fillna(0)

# Force binary (just in case): threshold at 0.5 for floats
labels_df = (labels_df.astype(float) >= 0.5).astype(int)

print("Label counts (positives per class):")
print(labels_df.sum().rename("positives"))

# --- compute Pearson correlation matrix ---
corr = labels_df.corr(method="pearson")

# --- quick report: top positive/negative correlated pairs (excluding diagonal) ---
def top_pairs(cmat, top_k=10):
    mat = cmat.copy()
    np.fill_diagonal(mat.values, np.nan)
    stacked = mat.unstack().dropna()
    # keep each pair once (i,j) same as (j,i) -> take only pairs where index1 < index2
    pairs = []
    seen = set()
    for (a,b),v in stacked.items():
        key = tuple(sorted((a,b)))
        if key in seen: 
            continue
        seen.add(key)
        pairs.append((key[0], key[1], float(v)))
    df_pairs = pd.DataFrame(pairs, columns=["A","B","corr"]).sort_values("corr", ascending=False)
    return df_pairs

pairs_df = top_pairs(corr, top_k=50)
print("\nTop 8 positively correlated label pairs:")
print(pairs_df.head(8).to_string(index=False))
print("\nTop 8 negatively correlated label pairs:")
print(pairs_df.tail(8).sort_values("corr").to_string(index=False))

# --- plotting: heatmap with mask of upper triangle ---
mask = np.triu(np.ones_like(corr, dtype=bool))
plt.figure(figsize=figsize)
sns.set(style="white")
cmap = sns.diverging_palette(220, 10, as_cmap=True)  # blue <-> red

ax = sns.heatmap(
    corr,
    mask=mask,
    cmap=cmap,
    center=0.0,
    vmax=1.0,
    vmin=-1.0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.7, "label": "Pearson r"},
    annot=True,
    fmt=annot_fmt,
    annot_kws={"fontsize": 8}
)

ax.set_xticklabels(ax.get_xticklabels(), rotation=45, horizontalalignment='right')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
ax.set_title("Pairwise Pearson Correlation — 14 Diseases", fontsize=14)
plt.tight_layout()
plt.savefig(OUT_FIG, dpi=150)
plt.show()

print(f"Correlation plot saved to: {OUT_FIG}")


In [ ]:
# Count the number of positive cases per disease
disease_counts = train_df[label_columns].sum().sort_values(ascending=True)

print("Disease counts:\n", disease_counts)

plt.figure(figsize=(10,6))
disease_counts.plot(kind='barh')
plt.title("Number of positive samples per disease")
plt.xlabel("Count")
plt.ylabel("Disease")
plt.show()
